In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Importing Prerequisites**

In [0]:
%run "/Workspace/FMCG Project/setup/3. utilities"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Read Data from Bronze Layer**

In [0]:
df_bronze = spark.sql("SELECT * FROM fmcg.bronze.products")
#df_bronze = spark.read.table("fmcg.bronze.products")

In [0]:
display(df_bronze)

## **Transformations**

### **1. Duplicate handling**

In [0]:
df_dup = display(df_bronze.groupBy("product_id","product_name","category").agg(F.count("*")))

In [0]:
print("Count of rows before dropping duplicates: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates()
print("Count of rows after dropping duplicates: ", df_bronze.count())

### **2. Title case fix**

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "category", 
    F.when(F.col("category").isNull(), None)
    .otherwise(F.initcap("category"))
    )

In [0]:
df_silver.select("category").distinct().show()